# SFT 数据路径：模型怎么知道新样本从哪里开始

Greedy packing 会把多条训练样本装进同一个定长容器。Attention 随后必须知道：哪些 token 属于同一条样本，哪里开始了下一条样本。

当前实现使用一条很直接的约定：**每放入一条新样本，它的位置编号都从 0 重新开始。** 训练器看到位置编号再次变成 0，就知道一条新样本开始了；再根据相邻起点算出每条样本的长度。

本节沿着这条数据链路展开：

```text
原始对话
  → tokenizer 生成完整 token
  → 构造右移一位的 input / labels
  → packing 时为每条样本重新从 0 编位置号
  → trainer 从这些起点恢复每条样本的长度
  → block-causal mask 或 Varlen cu_seq
```

EOS 仍然是聊天模板里的消息结束符，但它不负责标记 packed sample 边界。

## 1. 先分清三种信息

一条 Wordle 训练样本可能包含 system、user、assistant 等多条消息。三种信息各做一件事：

| 信息 | 含义 | 能否标记 packed sample 边界 |
|---|---|---|
| EOS / `<\|im_end\|>` | 一条聊天消息结束 | 不能：同一样本里可能出现很多次 |
| `labels == -100` | 该位置不参与 loss | 不能：prompt 和 padding 都可能被忽略 |
| position 重新从 0 开始 | 一条新样本或 padding 段开始 | 能：当前实现以它恢复边界 |

后文所说的“样本”是一个完整的训练记录，不是其中的一条 user/assistant message。

### 1.1 为什么不能扫描 EOS

tokenizer 先生成 `full_tokens`，然后 DataLoader 构造 next-token prediction 需要的右移输入：

```python
input_ids = full_tokens[:-1]
label_ids = full_tokens[1:]
```

因此，样本最后一个 EOS 会进入 labels，却可能不在 input 中；同一样本中较早的 message EOS 又会保留在 input 中。扫描 `input == eos_id` 会同时漏掉真正的样本结尾，并把一条多轮样本错误拆成多段。

In [ ]:
# 教学用 token：一条样本包含两轮消息，因此 full_tokens 中有多个 EOS。
BOS, EOS = 0, 99
full_tokens = [BOS, 11, EOS, 21, EOS, 31, EOS, 41, EOS]
input_ids = full_tokens[:-1]
label_ids = full_tokens[1:]

print('full tokens: ', full_tokens)
print('input ids:   ', input_ids)
print('labels:      ', label_ids)
print('EOS in input:', [i for i, token in enumerate(input_ids) if token == EOS])
print('样本末尾 EOS 只在 labels 最后一个位置:', label_ids[-1] == EOS)

assert input_ids[-1] != EOS
assert label_ids[-1] == EOS


### 1.2 Packing：每条样本的位置号重新从 0 开始

下面把两条样本装入同一个长度为 10 的容器。样本 A 长 4，样本 B 长 3，余下 3 个位置用于 padding：

```text
token 类型:  A A A A | B B B | padding padding padding
position:    0 1 2 3 | 0 1 2 | 0       1       2
起点:        ^         ^       ^
```

再次出现 0 的位置就是下一段的开头。position 同时让每条真实样本的 RoPE 从 0 开始；当前实现也把 padding 作为一个独立段。

In [ ]:
EOS = 99
sample_a = [11, 12, 13, 14]
sample_b = [21, 22, 23]
padding = [EOS, EOS, EOS]

packed_input = sample_a + sample_b + padding
positions = list(range(len(sample_a))) + list(range(len(sample_b))) + list(range(len(padding)))
starts = [index for index, position in enumerate(positions) if position == 0]
ends = starts[1:] + [len(positions)]
lengths = [end - start for start, end in zip(starts, ends)]

print('input:    ', packed_input)
print('positions:', positions)
print('段起点:   ', starts)
print('段长度:   ', lengths)

assert starts == [0, 4, 7]
assert lengths == [4, 3, 3]


### 观察 packing 输出

训练器不需要猜哪个 EOS 是 message 结束、sample 结束或 padding。它只读 positions：

- 位置 0：样本 A 开始；
- 位置 4：position 再次为 0，样本 B 开始；
- 位置 7：position 再次为 0，padding 段开始。

相邻起点之差给出长度 `[4, 3, 3]`。前两段是真实样本，最后一段是 padding。当前 VarLen wrapper 会收到全部 10 个 token；它用这些段隔离 attention，但没有在 Q/K/V 前删除 padding token。

## 2. Trainer：从样本起点得到累计长度

对一个 batch，trainer 把每行的 positions 展平，再找出所有重新从 0 开始的位置：

```text
positions [B, S]
  → 找出每个 position 为 0 的位置
  → 展平时加上每行的 batch offset
  → 在末尾补上 B*S
  → cu_seq = [0, L₁, L₁+L₂, ..., B*S]
```

实现代码中的判断写作 `positions == 0`。这只是上面“位置号重新从 0 开始”的代码表达，不需要读者把它当成另一套概念。

当 attention 配置为 block-causal 时，同一批起点有两种用法：

- dense SDPA：转换成 `[B,1,S,S]` 的同样本 causal mask；
- NPU VarLen：转换成 flattened `VarlenMetadata`，再交给 TND kernel。

### 2.1 演示：从 positions 到 `cu_seq`

下面用一小段代码复现当前实现的规则：先找出位置编号重新变成 0 的地方，再把整个 batch 的末尾补上。`cu_seq` 是把整个 batch 展平后的一个累计向量，不是每行各有一个嵌套列表。

In [ ]:
import torch

# Row 0: 长度 3、3、2；Row 1: 长度 2、4、2。
toy_positions = torch.tensor([
    [0, 1, 2, 0, 1, 2, 0, 1],
    [0, 1, 0, 1, 2, 3, 0, 1],
], dtype=torch.long)

sample_starts = toy_positions.eq(0).flatten().nonzero(as_tuple=True)[0]
cu_seq = torch.cat([
    sample_starts,
    torch.tensor([toy_positions.numel()], dtype=torch.long),
])
max_sample_length = int(torch.diff(cu_seq).max())

print('cu_seq:', cu_seq.tolist())
print('max sample length:', max_sample_length)

assert cu_seq.tolist() == [0, 3, 6, 8, 10, 14, 16]
assert max_sample_length == 4


## 3. TorchTitan 代码实现导览

上面的规则对应 TorchTitan 的真实数据路径（版本以第 2.03 节锁定的 commit 为准）：

```text
torchtitan/hf_datasets/text_datasets.py
  ChatDataset.__iter__ / _iter_greedy_packed
    → 生成 input_ids、labels、positions
torchtitan_npu/patches/torchtitan/chat_dataset.py
    → 根据 attention.mask_type 选择 greedy 或 non-greedy
torchtitan_npu/patches/torchtitan/trainer_post_dataloading_process.py
    → 从 positions == 0 的起点生成 block mask / VarlenMetadata
torchtitan_npu/attention/*
    → BSND → TND，调用 npu_fusion_attention_v3(sparse_mode=7)
```

`_iter_greedy_packed` 把每条 tokenized sample 追加到 buffer，在样本开头写入从 0 开始的 positions，容器放不下时补齐 padding 并返回固定长度张量。trainer 再寻找 `positions.eq(0)`，在末尾追加 `B*S`，得到 `cu_seq`。因此同一条多轮 Wordle 对话中的 `<|im_end|>` 不会被错误拆开。

若 attention backend 只支持整段 causal，patch 会关闭 greedy packing；只有声明 `block_causal` 或 VarLen metadata 时才会复用 container。positions 表示边界，labels 只表示 loss，mask/metadata 才负责文档隔离。

In [ ]:
# 在已准备好的 torchtitan-npu workspace 中快速定位实现；找不到时只需设置 TORCHTITAN_ROOT。
from pathlib import Path
import os
root = Path(os.environ.get('TORCHTITAN_ROOT', '/mnt/workspace/torchtitan-npu'))
paths = [
    root / 'torchtitan/hf_datasets/text_datasets.py',
    root / 'torchtitan_npu/patches/torchtitan/chat_dataset.py',
    root / 'torchtitan_npu/patches/torchtitan/trainer_post_dataloading_process.py',
]
for path in paths:
    print(f'{path}: {"OK" if path.is_file() else "missing"}')
    if path.is_file():
        text = path.read_text(errors='replace')
        for needle in ('_iter_greedy_packed', 'positions', 'cu_seq'):
            print(f'  {needle}: {text.count(needle)} occurrence(s)')


## 4. 完整链路与总结

```text
ChatDataset
  → 每条 packed sample 的 position 从 0 开始
  → trainer 找到所有样本起点
  → dense block mask 或 VarlenMetadata
  → NPUVarlenAttention: BSND → TND
  → cu_seq → actual_seq_qlen / actual_seq_kvlen
  → npu_fusion_attention_v3(sparse_mode=7)
```

三个要点：

1. EOS 是 message 结束符，不是 packed sample 边界；
2. 每条样本的位置号重新从 0 开始，训练器据此恢复边界；
3. labels 只控制 loss，既不能标记边界，也不能阻止跨样本 attention。

## 练习

1. 一条多轮样本中出现多个 EOS。为什么不能把每个 EOS 都当成一条新样本的边界？

2. （单选题）当前 trainer 根据什么判断一条新样本开始？
   A. `label == IGNORE_INDEX`
   B. position 编号重新变成 0
   C. `input == eos_id`
   D. attention 输出为 0

3. 给定 positions `[0,1,2,0,1,0,1,2]`，写出三段的起点、长度和 `cu_seq`。

4. （判断题）把某个 token 的 label 写成 `-100`，会阻止其他 token 读取它的 K/V。

In [ ]:
!cat ./answer/05.04_answer.txt
